# Demo: a multi-layer perceptron from scratch, then depth, width and activation

Each section calls the functions in `part1_mlp_scratch.py` and `part2_sklearn_exploration.py`
and shows what they compute; the notebook has no model code of its own. Running the two scripts
prints the same numbers and saves the same figures as files.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import part1_mlp_scratch as part1
import part2_sklearn_exploration as part2
from common import generate_moons, generate_spirals

# the scripts select the file-only Agg backend; switch back so figures show in the notebook
%matplotlib inline


def show(fig):
    display(fig)
    plt.close(fig)


Xm_train, ym_train, Xm_test, ym_test = generate_moons()
Xs_train, ys_train, Xs_test, ys_test = generate_spirals()
print("Two-Moons   train", Xm_train.shape, "test", Xm_test.shape)
print("Two-Spirals train", Xs_train.shape, "test", Xs_test.shape)

## Part 1: gradient check

Analytic gradients against central finite differences on five training rows, with H = 3 and
eps = 1e-5. The largest relative error has to be below 1e-6.

In [ ]:
for activation in part1.ACTIVATIONS:
    start, smallest_z, errors = part1.gradient_check(Xm_train, ym_train, activation)
    print(f"{activation}: training rows {start}-{start + 4}, smallest |z| = {smallest_z:.4f}")
    for name, error in errors.items():
        print(f"    d{name}  {error:.3e}")

## Part 1: Table 2

`ScratchMLP` on Two-Moons for H = 1, 2, 8, 32 and both activations: 2000 epochs of full-batch
gradient descent, learning rate 0.1, L2 penalty 1e-4.

In [ ]:
models = {}
print(f"{'activation':<10} {'H':>2} {'final loss':>10} {'train %':>8} {'test %':>7}")
for activation in part1.ACTIVATIONS:
    for H in part1.HIDDEN_SIZES:
        model = part1.ScratchMLP(H, activation).fit(Xm_train, ym_train, part1.SNAPSHOT_EPOCHS)
        models[activation, H] = model
        train = part1.pct(part1.accuracy(ym_train, model.predict(Xm_train)))
        test = part1.pct(part1.accuracy(ym_test, model.predict(Xm_test)))
        print(f"{activation:<10} {H:>2} {model.loss_history_[-1]:>10.4f} {train:>8} {test:>7}")

The training loss of the two H = 8 models, epoch by epoch (`loss_history_`).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
for activation, color in zip(part1.ACTIVATIONS, part1.CLASS_COLORS, strict=True):
    losses = models[activation, 8].loss_history_
    ax.plot(range(1, len(losses) + 1), losses, color=color, linewidth=2, label=activation)
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("ScratchMLP with H = 8 on Two-Moons")
ax.grid(color="#e6e6e3", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
show(fig)

## Part 1: decision boundary at H = 8

The boundary after 10, 200 and 2000 epochs, from the parameter snapshots that `fit` stores.
Leaky ReLU on top, tanh below.

In [ ]:
show(part1.boundary_figure({a: models[a, 8] for a in part1.ACTIVATIONS}, Xm_train, ym_train))

## Part 2.1: 36 hidden nodes, depth against width

Eight `MLPClassifier` models on Two-Spirals: the 36 nodes split over 1, 2, 3 or 6 layers, with
`logistic` and `relu`. The settings are `part2.SETTINGS` (Table 1 of the handout).

In [ ]:
print(part2.SETTINGS)
nets = {}
print(
    f"{'architecture':<20} {'activation':<10} {'train %':>8} {'test %':>7} {'loss':>8} {'norm':>9}"
)
for arch in part2.ARCHITECTURES:
    for activation in part2.ACTIVATIONS:
        clf = part2.train(arch, activation, Xs_train, ys_train)
        nets[arch, activation] = clf
        train, test = (
            part2.pct(clf.score(Xs_train, ys_train)),
            part2.pct(clf.score(Xs_test, ys_test)),
        )
        norm = np.linalg.norm(clf.coefs_[0], "fro")
        print(f"{arch!s:<20} {activation:<10} {train:>8} {test:>7} {clf.loss_:>8.4f} {norm:>9.4f}")

In [ ]:
show(part2.depth_vs_width_figure(nets, Xs_train, ys_train))

Loss curves of the deepest architecture, `(6, 6, 6, 6, 6, 6)`, for the two activations.

In [ ]:
show(part2.loss_curve_figure(nets))

## Part 2.2: width sweep

Two hidden layers of width M = 2, 4, 16, 64. P is the number of trainable parameters,
M^2 + 5M + 1.

In [ ]:
print(f"{'architecture':<12} {'P':>5} {'activation':<10} {'train %':>8} {'test %':>7}")
for M in part2.WIDTHS:
    for activation in part2.ACTIVATIONS:
        clf = part2.train((M, M), activation, Xs_train, ys_train)
        train, test = (
            part2.pct(clf.score(Xs_train, ys_train)),
            part2.pct(clf.score(Xs_test, ys_test)),
        )
        print(f"{(M, M)!s:<12} {part2.n_parameters(clf):>5} {activation:<10} {train:>8} {test:>7}")

## Part 2.2: the boundary of (16, 16) as it trains

One `partial_fit` call per epoch for 1000 epochs, with the boundary drawn after epochs 1, 25,
100, 300 and 1000. `logistic` on top, `relu` below.

In [ ]:
snapshots = part2.boundary_evolution(Xs_train, ys_train)
show(part2.evolution_figure(snapshots, Xs_train, ys_train))